In [3]:
%pip install -q sentence-transformers faiss-cpu transformers accelerate

In [4]:

import re
from pathlib import Path

file_path = Path("physics_212_syllabus.txt")

if not file_path.exists():
    from google.colab import files
    files.upload()

if not file_path.exists():
    raise FileNotFoundError("Upload physics_212_syllabus.txt first.")

syllabus_text = file_path.read_text(encoding="utf-8-sig")

def paragraph_split(syllabus_text):
    paragraphs = [
        p.strip()
        for p in re.split(r"\n\s*\n", syllabus_text)
        if p.strip()
    ]
    return paragraphs

paragraphs = paragraph_split(syllabus_text)

print("Paragraphs:", len(paragraphs))
print("Total words:", len(syllabus_text.split()))


# Strategy 1: Fixed-Length Chunking

def fixed_length_chunking(syllabus_text, max_chars_per_chunk=450):
    paragraphs = paragraph_split(syllabus_text)
    chunks = []

    for para in paragraphs:
        sentences = [
            s.strip()
            for s in re.split(r"(?<=[.!?])\s+", para)
            if s.strip()
        ]

        current_chunk = ""

        for sentence in sentences:
            if current_chunk and len(current_chunk) + len(sentence) + 1 > max_chars_per_chunk:
                chunks.append(current_chunk)
                current_chunk = sentence
            else:
                current_chunk = (
                    current_chunk + " " + sentence
                    if current_chunk else sentence
                )

        if current_chunk:
            chunks.append(current_chunk)

    return chunks


fixed_chunks = fixed_length_chunking(syllabus_text)

print("\nStrategy 1: Fixed-Length Chunking")
print("Number of chunks:", len(fixed_chunks))
print("Sample:\n", fixed_chunks[0])


# Strategy 2: Overlapping Paragraph Chunking

def overlapping_paragraph_chunking(syllabus_text):
    paragraphs = paragraph_split(syllabus_text)
    chunks = []

    for i in range(len(paragraphs) - 1):
        combined = paragraphs[i] + "\n\n" + paragraphs[i + 1]
        chunks.append(combined)

    return chunks


overlapping_chunks = overlapping_paragraph_chunking(syllabus_text)

print("\nStrategy 2: Overlapping Paragraph Chunking")
print("Number of chunks:", len(overlapping_chunks))
print("Sample:\n", overlapping_chunks[0])


# Strategy 3: Header-Based Strategic Chunking

def header_chunking(syllabus_text, header_pattern=r"^###\s+"):
    paragraphs = paragraph_split(syllabus_text)
    chunks = []
    current_chunk = []

    for para in paragraphs:
        if re.match(header_pattern, para):
            if current_chunk:
                chunks.append("\n\n".join(current_chunk))
                current_chunk = []

        current_chunk.append(para)

    if current_chunk:
        chunks.append("\n\n".join(current_chunk))

    return chunks


header_chunks = header_chunking(syllabus_text)

print("\nStrategy 3: Header-Based Chunking")
print("Number of chunks:", len(header_chunks))
print("Sample:\n", header_chunks[0])


Saving physics_212_syllabus.txt to physics_212_syllabus.txt
Paragraphs: 30
Total words: 2946

Strategy 1: Fixed-Length Chunking
Number of chunks: 63
Sample:
 ### 1. Course Overview & Objectives
Welcome to Physics 212, an intensive and comprehensive exploration into the fundamental principles of electricity and magnetism. Throughout this course, students will investigate the physical laws governing electric charges, electric fields, magnetic fields, electromagnetic induction, and Maxwell's equations.

Strategy 2: Overlapping Paragraph Chunking
Number of chunks: 29
Sample:
 ### 1. Course Overview & Objectives
Welcome to Physics 212, an intensive and comprehensive exploration into the fundamental principles of electricity and magnetism. Throughout this course, students will investigate the physical laws governing electric charges, electric fields, magnetic fields, electromagnetic induction, and Maxwell's equations. This course is designed primarily for engineering and physical science maj


## Step 1.2: Text Representation and Chunking

### Paragraph Definition

We define paragraphs using blank lines in the syllabus text. Each paragraph contains a section of course information, such as grading, assignments, or exams.

### Fixed-Length Chunking

We use a target chunk size of 450 characters. The text is divided at sentence boundaries to avoid splitting sentences.

Each paragraph is processed separately so unrelated sections are not combined.

### Overlapping Paragraph Chunking

We combine adjacent paragraphs with overlap:

- Chunk 1: Paragraph 1 + Paragraph 2
- Chunk 2: Paragraph 2 + Paragraph 3
- Chunk 3: Paragraph 3 + Paragraph 4

This preserves information between consecutive chunks. However, it can also combine adjacent sections that discuss different topics.

### Hybrid/Strategic Chunking

We use the syllabus's Markdown headings to identify section boundaries. Each heading and its associated text form a separate chunk.

This keeps related information together and avoids combining unrelated syllabus sections.

### Comparison

Fixed-length chunking produces shorter chunks with relatively consistent sizes.

Overlapping paragraph chunking preserves more context but introduces repeated information.

Header-based chunking follows the organization of the syllabus and keeps complete topics together.


In [5]:
# Install required libraries if not already present in your Colab environment
# !pip install sentence-transformers faiss-cpu
!pip install faiss-cpu sentence-transformers transformers accelerate
import numpy as np
from sentence_transformers import SentenceTransformer
import faiss

def build_multi_model_vector_indices(chunks: list[str]) -> dict:
    if not chunks:
        raise ValueError("The input list of chunks is empty.")

    # 1. Define Hugging Face models for Small, Medium, and Large tiers
    model_configs = {
        "small": "sentence-transformers/all-MiniLM-L6-v2",    # Fast & Lightweight (~90MB)
        "medium": "sentence-transformers/all-mpnet-base-v2",  # High quality, balanced (~420MB)
        "large": "BAAI/bge-large-en-v1.5"                     # SOTA high-capacity model (~1.3GB)
    }

    searchable_structures = {}

    for size, model_name in model_configs.items():
        print(f"[{size.upper()}] Loading model: {model_name}...")
        model = SentenceTransformer(model_name, device="cpu")

        # Step 1: Convert each chunk into embeddings
        print(f"[{size.upper()}] Encoding {len(chunks)} chunks into embeddings...")
        embeddings = model.encode(chunks, show_progress_bar=True, convert_to_numpy=True)

        # Normalize vectors for cosine similarity
        faiss.normalize_L2(embeddings)
        embedding_dim = embeddings.shape[1]

        # Step 2: Store embeddings in a searchable FAISS structure (Inner Product / Cosine Similarity)
        index = faiss.IndexFlatIP(embedding_dim)
        index.add(embeddings.astype(np.float32))

        searchable_structures[size] = {
            "model_name": model_name,
            "model": model,
            "index": index,
            "chunks": chunks,
            "dimension": embedding_dim
        }
        print(f"[{size.upper()}] Successfully indexed vectors with dimension {embedding_dim}.\n")

    return searchable_structures


vector_stores = build_multi_model_vector_indices(fixed_chunks)

for size, store in vector_stores.items():
    print("Model:", size)
    print("Dimensions:", store["dimension"])
    print("Indexed chunks:", store["index"].ntotal)
    print()


[SMALL] Loading model: sentence-transformers/all-MiniLM-L6-v2...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

... (39 outputs left out)

config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]

[LARGE] Encoding 63 chunks into embeddings...


Batches:   0%|          | 0/2 [00:00<?, ?it/s]

[LARGE] Successfully indexed vectors with dimension 1024.

Model: small
Dimensions: 384
Indexed chunks: 63

Model: medium
Dimensions: 768
Indexed chunks: 63

Model: large
Dimensions: 1024
Indexed chunks: 63



In [6]:
import numpy as np

def retrieve_relevant_chunks(vector_stores: dict, model_tier: str, query: str, top_k: int = 3) -> list[dict]:
    """
    Performs similarity search and top-k retrieval for a given query.

    Parameters:
        vector_stores (dict): The dictionary returned by build_multi_model_vector_indices.
        model_tier (str): Which model to use ('small', 'medium', or 'large').
        query (str): The search query text.
        top_k (int): Number of top relevant chunks to retrieve.

    Returns:
        list[dict]: A ranked list of dictionaries containing similarity scores and chunk text.
    """
    # Validate model tier
    if model_tier not in vector_stores:
        raise ValueError(f"Invalid tier '{model_tier}'. Choose from: {list(vector_stores.keys())}")

    store = vector_stores[model_tier]
    model = store["model"]
    index = store["index"]
    chunks = store["chunks"]

    # 1. Convert Query to Embedding
    # Encode the query string into a vector array
    query_embedding = model.encode([query], convert_to_numpy=True)

    # 2. Similarity Search Prep (L2 Normalization)
    # Since we used IndexFlatIP (Inner Product) with normalized vectors during indexing,
    # we must normalize the query vector to compute true cosine similarity scores.
    faiss.normalize_L2(query_embedding)

    # 3. Perform Similarity Search
    # index.search returns distance/score matrices and corresponding array indices
    distances, indices = index.search(query_embedding.astype(np.float32), top_k)

    # 4. Top-K Retrieval of Relevant Chunks
    retrieved_results = []
    for rank, (score, idx) in enumerate(zip(distances[0], indices[0])):
        if idx == -1:  # FAISS returns -1 if index contains fewer items than top_k
            continue

        retrieved_results.append({
            "rank": rank + 1,
            "similarity_score": float(score), # Higher score means higher similarity (range ~[-1, 1])
            "chunk_index": int(idx),
            "content": chunks[idx]
        })

    return retrieved_results


query_text = "What is the penalty for submitting homework late?"

top_results = retrieve_relevant_chunks(
    vector_stores,
    model_tier="medium",
    query=query_text,
    top_k=3
)

print("Question:", query_text)

for result in top_results:
    print("\nRank:", result["rank"])
    print("Similarity:", round(result["similarity_score"], 4))
    print("Chunk index:", result["chunk_index"])
    print("Retrieved text:")
    print(result["content"])


Question: What is the penalty for submitting homework late?

Rank: 1
Similarity: 0.6748
Chunk index: 23
Retrieved text:
All homework must be submitted digitally through the online portal or handed in physically as instructed. Late submissions will incur a severe penalty of twenty percent per day late, and no homework will be accepted more than seventy-two hours past the original deadline.

Rank: 2
Similarity: 0.4467
Chunk index: 22
Retrieved text:
To reinforce classroom learning, there will be exactly one major homework assignment due every week. In strict accordance with course guidelines, these assignments are always due on Thursday right before class begins, specifically at 12:00 PM. Please note that weekly assignments are explicitly excluded during the first week of the semester and the final week of the semester.

Rank: 3
Similarity: 0.4059
Chunk index: 21
Retrieved text:
Because the semester moves at a rapid pace, missing a single session can create significant gaps in your under

In [7]:

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

GENERATION_MODEL_ID = "Qwen/Qwen3-4B-Instruct-2507"

generation_tokenizer = None
generation_model = None


def load_generation_model():
    global generation_tokenizer, generation_model

    if generation_model is None:
        print("Loading:", GENERATION_MODEL_ID)

        generation_tokenizer = AutoTokenizer.from_pretrained(
            GENERATION_MODEL_ID
        )

        if torch.cuda.is_available():
            dtype = (
                torch.bfloat16
                if torch.cuda.is_bf16_supported()
                else torch.float16
            )
        else:
            dtype = torch.float32

        generation_model = AutoModelForCausalLM.from_pretrained(
            GENERATION_MODEL_ID,
            torch_dtype=dtype,
            device_map="auto"
        )

        generation_model.eval()

    return generation_tokenizer, generation_model


def generate_rag_response(query, retrieved_chunks):

    tokenizer, model = load_generation_model()

    context_text = "\n\n".join(
        f"Syllabus Section {i + 1}:\n{chunk['content']}"
        for i, chunk in enumerate(retrieved_chunks)
    )

    system_instruction = (
        "You are an AI teaching assistant for Physics 212. "
        "Answer the student's question using only the retrieved "
        "syllabus information. If the information is not present, "
        "say that it is not available in the provided context. "
        "Do not invent course policies or deadlines."
    )

    user_prompt = (
        f"Retrieved syllabus context:\n{context_text}\n\n"
        f"Question: {query}"
    )

    messages = [
        {"role": "system", "content": system_instruction},
        {"role": "user", "content": user_prompt}
    ]

    inputs = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors="pt"
    ).to(model.device)

    with torch.inference_mode():
        outputs = model.generate(
            **inputs,
            max_new_tokens=256,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )

    generated_ids = outputs[0][inputs["input_ids"].shape[-1]:]

    response = tokenizer.decode(
        generated_ids,
        skip_special_tokens=True
    ).strip()

    return response


In [8]:

print("=== RAG SYSTEM TEST ===")

query_text = "What is the penalty for submitting homework late?"

top_results = retrieve_relevant_chunks(
    vector_stores,
    model_tier="medium",
    query=query_text,
    top_k=3
)

print("\nQuestion:")
print(query_text)

print("\nRetrieved Chunks:")

for result in top_results:
    print(f"\nRank {result['rank']}")
    print(f"Similarity: {result['similarity_score']:.4f}")
    print(result["content"])

print("\nGenerating answer...")

final_answer = generate_rag_response(
    query_text,
    top_results
)

print("\nFinal Answer:")
print(final_answer)


=== RAG SYSTEM TEST ===

Question:
What is the penalty for submitting homework late?

Retrieved Chunks:

Rank 1
Similarity: 0.6748
All homework must be submitted digitally through the online portal or handed in physically as instructed. Late submissions will incur a severe penalty of twenty percent per day late, and no homework will be accepted more than seventy-two hours past the original deadline.

Rank 2
Similarity: 0.4467
To reinforce classroom learning, there will be exactly one major homework assignment due every week. In strict accordance with course guidelines, these assignments are always due on Thursday right before class begins, specifically at 12:00 PM. Please note that weekly assignments are explicitly excluded during the first week of the semester and the final week of the semester.

Rank 3
Similarity: 0.4059
Because the semester moves at a rapid pace, missing a single session can create significant gaps in your understanding. Please make every effort to arrive on time, p

config.json:   0%|          | 0.00/727 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/9.38k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 11.4MB            

... (4 outputs left out)

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/238 [00:00<?, ?B/s]


Final Answer:
The penalty for submitting homework late is twenty percent per day late. No homework will be accepted more than seventy-two hours past the original deadline.


In [9]:

query = "How much is the final exam worth?"

for model_size in ["small", "medium", "large"]:

    print(f"\n=== {model_size.upper()} MODEL ===")

    results = retrieve_relevant_chunks(
        vector_stores,
        model_tier=model_size,
        query=query,
        top_k=3
    )

    for result in results:
        print("Rank:", result["rank"])
        print("Similarity:", round(result["similarity_score"], 4))
        print("Content:", result["content"])
        print("-" * 50)



=== SMALL MODEL ===
Rank: 1
Similarity: 0.5972
Content: Like the midterm, the final exam will evaluate both conceptual reasoning and advanced quantitative problem-solving capabilities. Preparing for this exam requires a systematic review of all past homework assignments, mid-term corrections, and lecture notes. Success on this final exam is critical, as it accounts for thirty-five percent of your overall course grade.
--------------------------------------------------
Rank: 2
Similarity: 0.5472
Content: The mid-semester examination administered during week eight will count for thirty percent, while the comprehensive final examination during week sixteen will represent the remaining thirty-five percent of your total grade. Letter grades will be assigned based on standard university grading scales, with absolute percentage thresholds outlined in the course syllabus document.
--------------------------------------------------
Rank: 3
Similarity: 0.5082
Content: The final exam takes place

# Part 2: System Experiments

## 2.1 Test Queries

We selected five realistic questions about Physics 212 to evaluate retrieval and answer quality across different embedding models and chunking strategies.

In [10]:
test_queries = [
    "When are weekly homework assignments due?",
    "What happens if I submit homework late?",
    "How much of my grade is the final exam worth?",
    "When and where is the midterm exam?",
    "When does the instructor hold office hours?"
]

expected_answers = [
    "Thursday at 12:00 PM, except during the first and final weeks.",
    "20% penalty per day, and submissions are not accepted after 72 hours.",
    "35% of the final grade.",
    "Thursday of Week 8 at 12:00 PM in Lecture Hall 210.",
    "Tuesday and Thursday afternoons after class, or by appointment in Room 402."
]

for i, question in enumerate(test_queries, 1):
    print(i, question)

1 When are weekly homework assignments due?
2 What happens if I submit homework late?
3 How much of my grade is the final exam worth?
4 When and where is the midterm exam?
5 When does the instructor hold office hours?


In [11]:
import time
import gc
import pandas as pd
import numpy as np
import faiss
from pathlib import Path

chunking_strategies = {
    "fixed": fixed_chunks,
    "overlapping": overlapping_chunks,
    "header": header_chunks
}

model_names = {
    "small": "sentence-transformers/all-MiniLM-L6-v2",
    "medium": "sentence-transformers/all-mpnet-base-v2",
    "large": "BAAI/bge-large-en-v1.5"
}

results_file = Path("part2_results.csv")

print("Number of configurations:", len(chunking_strategies) * len(model_names))
print("Total tests:", len(chunking_strategies) * len(model_names) * len(test_queries))

Number of configurations: 9
Total tests: 45


In [12]:
def create_experiment_store(model, chunks):
    embeddings = model.encode(
        chunks,
        convert_to_numpy=True,
        show_progress_bar=False
    ).astype(np.float32)

    faiss.normalize_L2(embeddings)

    index = faiss.IndexFlatIP(embeddings.shape[1])
    index.add(embeddings)

    return {
        "model": model,
        "index": index,
        "chunks": chunks
    }

In [13]:
if results_file.exists():
    experiment_results = pd.read_csv(results_file).to_dict("records")
else:
    experiment_results = []

completed = {
    (r["Embedding Model"], r["Chunking Strategy"], r["Query"])
    for r in experiment_results
}

for model_size, model_name in model_names.items():

    model = vector_stores[model_size]["model"]

    for strategy_name, chunks in chunking_strategies.items():

        print(f"\nTesting {model_size} + {strategy_name}")

        store = create_experiment_store(model, chunks)

        stores = {
            model_size: store
        }

        for query in test_queries:

            key = (model_size, strategy_name, query)

            if key in completed:
                continue

            start_time = time.perf_counter()

            retrieved = retrieve_relevant_chunks(
                stores,
                model_tier=model_size,
                query=query,
                top_k=3
            )

            answer = generate_rag_response(
                query,
                retrieved
            )

            latency = time.perf_counter() - start_time

            retrieved_text = "\n\n".join(
                f"Rank {r['rank']} | Score {r['similarity_score']:.4f}\n"
                f"{r['content']}"
                for r in retrieved
            )

            experiment_results.append({
                "Embedding Model": model_size,
                "Chunking Strategy": strategy_name,
                "Query": query,
                "Retrieved Context Quality": "",
                "Answer Quality": "",
                "Latency": round(latency, 3),
                "Notes": "",
                "Retrieved Chunks": retrieved_text,
                "Final Answer": answer
            })

            pd.DataFrame(experiment_results).to_csv(
                results_file,
                index=False
            )

            completed.add(key)

            print("Question:", query)
            print("Retrieved Chunks:\n", retrieved_text)
            print("Generated Answer:", answer)
            print("Latency:", round(latency, 3), "seconds")
            print("-" * 60)

        del store, stores
        gc.collect()

print("\nCompleted tests:", len(experiment_results))


Testing small + fixed
Question: When are weekly homework assignments due?
Retrieved Chunks:
 Rank 1 | Score 0.8619
To reinforce classroom learning, there will be exactly one major homework assignment due every week. In strict accordance with course guidelines, these assignments are always due on Thursday right before class begins, specifically at 12:00 PM. Please note that weekly assignments are explicitly excluded during the first week of the semester and the final week of the semester.

Rank 2 | Score 0.5925
All homework must be submitted digitally through the online portal or handed in physically as instructed. Late submissions will incur a severe penalty of twenty percent per day late, and no homework will be accepted more than seventy-two hours past the original deadline.

Rank 3 | Score 0.5681
Homework assignment number twelve will be due on Thursday of Week 14 right before class at 12:00 PM, concluding our weekly homework submission cycle for the semester.
Generated Answer: Wee

In [14]:
results_df = pd.read_csv("part2_results.csv")

display(results_df[
    [
        "Embedding Model",
        "Chunking Strategy",
        "Query",
        "Retrieved Context Quality",
        "Answer Quality",
        "Latency",
        "Notes"
    ]
])

print("Total rows:", len(results_df))

,Embedding Model,Chunking Strategy,Query,Retrieved Context Quality,Answer Quality,Latency,Notes
0,small,fixed,When are weekly homework assignments due?,NaN,NaN,5.684,NaN
1,small,fixed,What happens if I submit homework late?,NaN,NaN,3.084,NaN
2,small,fixed,How much of my grade is the final exam worth?,NaN,NaN,2.164,NaN
3,small,fixed,When and where is the midterm exam?,NaN,NaN,3.546,NaN
4,small,fixed,When does the instructor hold office hours?,NaN,NaN,3.276,NaN
5,small,overlapping,When are weekly homework assignments due?,NaN,NaN,7.306,NaN
6,small,overlapping,What happens if I submit homework late?,NaN,NaN,6.074,NaN
7,small,overlapping,How much of my grade is the final exam worth?,NaN,NaN,4.787,NaN
8,small,overlapping,When and where is the midterm exam?,NaN,NaN,5.780,NaN
9,small,overlapping,When does the instructor hold office hours?,NaN,NaN,4.472,NaN


Total rows: 45


In [16]:
import pandas as pd

results_df = pd.read_csv("part2_results.csv")

print("Total experiments:", len(results_df))
display(results_df.head())

Total experiments: 45


,Embedding Model,Chunking Strategy,Query,Retrieved Context Quality,Answer Quality,Latency,Notes,Retrieved Chunks,Final Answer
0,small,fixed,When are weekly homework assignments due?,2,2,5.684,Homework deadline policy retrieved (Thursday 1...,Rank 1 | Score 0.8619\nTo reinforce classroom ...,Weekly homework assignments are due on Thursda...
1,small,fixed,What happens if I submit homework late?,2,2,3.084,Late-work policy retrieved (20% per day; 72-ho...,Rank 1 | Score 0.7629\nAll homework must be su...,"If you submit homework late, a severe penalty ..."
2,small,fixed,How much of my grade is the final exam worth?,2,2,2.164,Grading/final-exam section retrieved (final = ...,Rank 1 | Score 0.6348\nThe mid-semester examin...,The final exam is worth thirty-five percent of...
3,small,fixed,When and where is the midterm exam?,2,2,3.546,"Midterm guidance retrieved (Week 8, Thursday 1...",Rank 1 | Score 0.7259\n### 12. Mid-Term Examin...,The midterm exam is on Thursday at 12:00 PM in...
4,small,fixed,When does the instructor hold office hours?,2,2,3.276,Instructor office-hours section retrieved (Tue...,Rank 1 | Score 0.6741\n### 3. Instructor Infor...,The instructor holds office hours every Tuesda...


In [17]:
embedding_comparison = results_df.groupby("Embedding Model")[
    ["Retrieved Context Quality", "Answer Quality", "Latency"]
].mean()

display(embedding_comparison.round(2))

,Retrieved Context Quality,Answer Quality,Latency
Embedding Model,,,
large,1.87,1.87,5.47
medium,1.73,1.67,4.49
small,1.87,1.87,4.43


## 2.3 Embedding Model Comparison

We compared three embedding models: all-MiniLM-L6-v2 (384 dimensions), all-mpnet-base-v2 (768 dimensions), and BGE-large-en-v1.5 (1024 dimensions). Each model was tested with three chunking strategies and five syllabus questions.

The small and large models achieved the same average retrieval quality score of 1.87/2 and answer quality score of 1.87/2. The medium model performed slightly worse, with a retrieval quality score of 1.73/2 and an answer quality score of 1.67/2.

Larger embeddings did not always perform better. Although the large model used 1024 dimensions, it did not improve the average quality scores compared with the small 384-dimensional model. It also had a higher average latency of 5.47 seconds, compared with 4.43 seconds for the small model.

One important failure occurred when retrieving instructor office hours. All three embedding models failed to retrieve the relevant information when overlapping paragraph chunking was used. This suggests that chunking strategy can have a significant effect on retrieval quality regardless of embedding size.

Overall, the small embedding model provided the best balance of answer quality and latency for our Physics 212 syllabus dataset. However, these findings are limited to the five queries and dataset used in our experiments.

In [18]:
chunking_comparison = results_df.groupby("Chunking Strategy")[
    ["Retrieved Context Quality", "Answer Quality", "Latency"]
].mean()

display(chunking_comparison.round(2))

,Retrieved Context Quality,Answer Quality,Latency
Chunking Strategy,,,
fixed,2.00,1.93,3.56
header,1.87,1.87,4.81
overlapping,1.60,1.60,6.01


In [19]:
example = results_df[
    (results_df["Embedding Model"] == "medium") &
    (results_df["Query"] == "When does the instructor hold office hours?")
]

for _, row in example.iterrows():
    print("=" * 70)
    print("Embedding Model:", row["Embedding Model"])
    print("Chunking Strategy:", row["Chunking Strategy"])
    print("Retrieved Context Quality:", row["Retrieved Context Quality"])
    print("Answer Quality:", row["Answer Quality"])
    print("\nRetrieved Chunks:")
    print(row["Retrieved Chunks"])
    print("\nFinal Answer:")
    print(row["Final Answer"])
    print()

Embedding Model: medium
Chunking Strategy: fixed
Retrieved Context Quality: 2
Answer Quality: 2

Retrieved Chunks:
Rank 1 | Score 0.5501
### 3. Instructor Information & Office Hours
Your instructor for this semester is Dr. Alexander Vance, Professor of Physics. Office hours will be held every Tuesday and Thursday afternoon following class, as well as by appointment in Room 402 of the Science and Engineering Building.

Rank 2 | Score 0.5492
### 10. Lecture Schedule & Punctuality
Physics 212 officially meets every Tuesday and Thursday throughout the fall semester. Our lecture and discussion session begins precisely at 12:00 PM and runs for one full hour, concluding at 1:00 PM in Lecture Hall 210. Punctuality is essential, as announcements and critical conceptual introductions typically occur during the first five minutes of class.

Rank 3 | Score 0.5384
The exam occurs during our regular Thursday class time slot at 12:00 PM in Lecture Hall 210, with no separate homework assignment due th

## 2.4 Chunking Strategy Comparison

We evaluated three chunking strategies: fixed-length chunking, overlapping paragraph chunking, and header-based chunking. Each strategy was tested with all three embedding models using the same five queries.

Fixed-length chunking performed best, achieving an average retrieval quality score of 2.00/2 and answer quality score of 1.93/2. Header-based chunking performed slightly worse, with both scores averaging 1.87/2. Overlapping paragraph chunking had the lowest average scores of 1.60/2 for both retrieval and answer quality.

Fixed-length chunking was effective because it divided the syllabus into shorter, focused pieces of text while preserving sentence and paragraph boundaries. This made it easier for the embedding models to identify relevant information without including too much unrelated context.

Overlapping paragraph chunking combined adjacent paragraphs, which preserved more surrounding context but sometimes introduced unnecessary information. This made retrieval less precise for certain questions.

For example, when using the medium embedding model to answer "When does the instructor hold office hours?", fixed-length chunking retrieved the correct office-hours section. Qwen correctly answered that office hours were held on Tuesday and Thursday afternoons following class, with appointments available in Room 402.

However, overlapping and header-based chunking failed to retrieve the office-hours information for the same question. Instead, they retrieved other syllabus sections, and Qwen incorrectly stated that office hours were not provided.

This demonstrates how chunking affects the entire RAG pipeline. When the retrieved text contains the correct facts, the generation model can produce an accurate response. When retrieval misses those facts, even the same generation model may produce an incorrect or incomplete answer.

Overall, fixed-length chunking was the most effective strategy for our dataset, providing the highest quality scores and lowest average latency of 3.56 seconds. These results suggest that shorter, focused chunks are better suited to our syllabus question-answering system.

In [20]:
assert len(results_df) == 45

required_columns = [
    "Embedding Model",
    "Chunking Strategy",
    "Query",
    "Retrieved Chunks",
    "Final Answer",
    "Retrieved Context Quality",
    "Answer Quality",
    "Latency",
    "Notes"
]

assert results_df[required_columns].notna().all().all()

results_df[required_columns].to_csv(
    "part2_evaluation_complete.csv",
    index=False
)

print("All 45 evaluations, including retrieved chunks, saved.")

All 45 evaluations, including retrieved chunks, saved.


## 2.5 Data Scaling Experiment

In [21]:
paragraphs = paragraph_split(syllabus_text)

small_text = "\n\n".join(paragraphs[:10])
large_text = syllabus_text

small_chunks = fixed_length_chunking(small_text)
large_chunks = fixed_length_chunking(large_text)

print("Small dataset:")
print("Paragraphs:", 10)
print("Chunks:", len(small_chunks))

print("\nLarge dataset:")
print("Paragraphs:", len(paragraphs))
print("Chunks:", len(large_chunks))

Small dataset:
Paragraphs: 10
Chunks: 22

Large dataset:
Paragraphs: 30
Chunks: 63


In [22]:
import time
import pandas as pd

datasets = {
    "Small": small_chunks,
    "Large": large_chunks
}

scaling_results = []

model = vector_stores["medium"]["model"]

for dataset_name, chunks in datasets.items():

    store = create_experiment_store(model, chunks)

    for query in test_queries:

        start = time.perf_counter()

        retrieved = retrieve_relevant_chunks(
            {"medium": store},
            model_tier="medium",
            query=query,
            top_k=3
        )

        latency = time.perf_counter() - start

        scaling_results.append({
            "Dataset": dataset_name,
            "Number of Chunks": len(chunks),
            "Query": query,
            "Latency": round(latency, 4),
            "Retrieved Chunks": "\n\n".join(
                f"Rank {r['rank']}: {r['content']}"
                for r in retrieved
            ),
            "Retrieval Quality": "",
            "Noise": "",
            "Notes": ""
        })

        print("=" * 60)
        print("Dataset:", dataset_name)
        print("Query:", query)

        for r in retrieved:
            print("\nRank:", r["rank"])
            print("Similarity:", round(r["similarity_score"], 4))
            print(r["content"])

    del store

scaling_df = pd.DataFrame(scaling_results)

display(scaling_df)

scaling_df.to_csv("scaling_results.csv", index=False)

Dataset: Small
Query: When are weekly homework assignments due?

Rank: 1
Similarity: 0.4549
### 8. Grading Policy & Weight Distribution
Your final course grade will be determined through a weighted combination of homework assignments, quizzes, a mid-semester examination, and a cumulative final examination. Weekly homework assignments account for twenty percent of your final grade. Regular online and in-class quizzes will contribute fifteen percent.

Rank: 2
Similarity: 0.4383
### 10. Lecture Schedule & Punctuality
Physics 212 officially meets every Tuesday and Thursday throughout the fall semester. Our lecture and discussion session begins precisely at 12:00 PM and runs for one full hour, concluding at 1:00 PM in Lecture Hall 210. Punctuality is essential, as announcements and critical conceptual introductions typically occur during the first five minutes of class.

Rank: 3
Similarity: 0.4152
Because the semester moves at a rapid pace, missing a single session can create significant ga

,Dataset,Number of Chunks,Query,Latency,Retrieved Chunks,Retrieval Quality,Noise,Notes
0,Small,22,When are weekly homework assignments due?,0.0927,Rank 1: ### 8. Grading Policy & Weight Distrib...,,,
1,Small,22,What happens if I submit homework late?,0.1115,Rank 1: Because the semester moves at a rapid ...,,,
2,Small,22,How much of my grade is the final exam worth?,0.1260,Rank 1: The mid-semester examination administe...,,,
3,Small,22,When and where is the midterm exam?,0.1129,Rank 1: ### 10. Lecture Schedule & Punctuality...,,,
4,Small,22,When does the instructor hold office hours?,0.1138,Rank 1: ### 3. Instructor Information & Office...,,,
5,Large,63,When are weekly homework assignments due?,0.0939,"Rank 1: To reinforce classroom learning, there...",,,
6,Large,63,What happens if I submit homework late?,0.1101,Rank 1: All homework must be submitted digital...,,,
7,Large,63,How much of my grade is the final exam worth?,0.1332,Rank 1: The mid-semester examination administe...,,,
8,Large,63,When and where is the midterm exam?,0.1090,Rank 1: The exam occurs during our regular Thu...,,,
9,Large,63,When does the instructor hold office hours?,0.1109,Rank 1: ### 3. Instructor Information & Office...,,,


## 2.5 Data Scaling Experiment

For this experiment, we compared a smaller dataset containing the first 10 paragraphs of the Physics 212 syllabus with the complete dataset containing 30 paragraphs. We used the same medium embedding model (all-mpnet-base-v2), fixed-length chunking, five test queries, and top-3 retrieval.

**Retrieval Quality:** The smaller dataset achieved an average retrieval quality score of 1.00/2, while the larger dataset achieved 2.00/2. The larger dataset performed better because it contained information missing from the smaller subset. For example, the smaller dataset failed to retrieve the homework deadline and late-submission policy, while the larger dataset correctly retrieved Thursday at 12:00 PM as the deadline and the 20% daily late penalty with a 72-hour cutoff.

**Noise:** The smaller dataset had an average noise score of 1.40/2, compared with 0.80/2 for the larger dataset. When relevant information was missing, the smaller dataset retrieved unrelated content about lecture attendance and general grading policies. The larger dataset generally retrieved more relevant information, although some unrelated or unnecessary chunks were still present.

**Latency:** Both datasets had nearly identical average retrieval latency of approximately 0.1114 seconds. Increasing the dataset from 10 to 30 paragraphs did not noticeably affect retrieval speed in this small experiment.

**Conclusion:** Increasing dataset size improved retrieval quality and reduced noise in our experiment. However, this improvement was largely due to the additional information available in the full syllabus. Because the smaller dataset excluded some relevant policies, the experiment does not establish that larger datasets are inherently more accurate or less noisy. With much larger datasets, additional irrelevant information could increase retrieval noise.

In [23]:
scaling_df = pd.read_csv("scaling_results.csv")

scaling_df["Retrieval Quality"] = pd.to_numeric(
    scaling_df["Retrieval Quality"]
)

scaling_df["Noise"] = pd.to_numeric(
    scaling_df["Noise"]
)

scaling_comparison = scaling_df.groupby("Dataset")[
    ["Retrieval Quality", "Noise", "Latency"]
].mean()

display(scaling_comparison.round(3))

,Retrieval Quality,Noise,Latency
Dataset,,,
Large,2.0,0.8,0.111
Small,1.0,1.4,0.111


# Part 3: Failure Analysis and Improvement

## 3.1 Failure Cases

In [25]:
import pandas as pd

results_df = pd.read_csv("part2_results.csv")

failures = [
    ("small", "overlapping",
     "When does the instructor hold office hours?"),

    ("medium", "header",
     "When does the instructor hold office hours?"),

    ("medium", "fixed",
     "When and where is the midterm exam?")
]

failure_rows = []

for model, strategy, query in failures:
    row = results_df[
        (results_df["Embedding Model"] == model) &
        (results_df["Chunking Strategy"] == strategy) &
        (results_df["Query"] == query)
    ].iloc[0]

    failure_rows.append(row)

    print("=" * 70)
    print("Model:", model)
    print("Chunking:", strategy)
    print("Question:", query)
    print("Retrieval Quality:", row["Retrieved Context Quality"])
    print("Answer Quality:", row["Answer Quality"])

    print("\nRetrieved Chunks:")
    print(row["Retrieved Chunks"])

    print("\nGenerated Answer:")
    print(row["Final Answer"])
    print()

failure_df = pd.DataFrame(failure_rows)

Model: small
Chunking: overlapping
Question: When does the instructor hold office hours?
Retrieval Quality: 0
Answer Quality: 0

Retrieved Chunks:
Rank 1 | Score 0.5237
### 10. Lecture Schedule & Punctuality
Physics 212 officially meets every Tuesday and Thursday throughout the fall semester. Our lecture and discussion session begins precisely at 12:00 PM and runs for one full hour, concluding at 1:00 PM in Lecture Hall 210. Punctuality is essential, as announcements and critical conceptual introductions typically occur during the first five minutes of class. Because the semester moves at a rapid pace, missing a single session can create significant gaps in your understanding. Please make every effort to arrive on time, prepared with your textbook, notebook, and calculator for active problem-solving engagement.

To reinforce classroom learning, there will be exactly one major homework assignment due every week. In strict accordance with course guidelines, these assignments are always d

### Failure 1: Small Embedding Model + Overlapping Chunking

**Question:** When does the instructor hold office hours?

**Expected Answer:** Office hours are held Tuesday and Thursday afternoons following class, with appointments available in Room 402.

**Actual Answer:** The model stated that the syllabus did not provide office-hours information.

**Problem:** The top three retrieved chunks contained information about lecture schedules, grading, and other course policies rather than the instructor's office-hours schedule. The model therefore gave an incorrect answer relative to the full syllabus.

### Failure 2: Medium Embedding Model + Header-Based Chunking

**Question:** When does the instructor hold office hours?

**Expected Answer:** Tuesday and Thursday afternoons following class, with appointments available in Room 402.

**Actual Answer:** The model stated that office-hours information was unavailable.

**Problem:** The retrieved chunks did not contain the instructor's office-hours policy. Instead, the system retrieved unrelated course information, resulting in an incorrect answer.

### Failure 3: Medium Embedding Model + Fixed-Length Chunking

**Question:** When and where is the midterm exam?

**Expected Answer:** Week 8, Thursday at 12:00 PM in Lecture Hall 210.

**Actual Answer:** The midterm exam is on Thursday at 12:00 PM in Lecture Hall 210.

**Problem:** The model correctly identified the time and location but omitted Week 8. The retrieved context contained all three facts, so this was an incomplete generation rather than a retrieval failure.

## 3.2 Root Cause Analysis

### Failure 1: Small Model + Overlapping Chunking

The likely cause was a combination of chunking and retrieval ranking. Overlapping paragraph chunking combined adjacent syllabus paragraphs, creating longer passages that sometimes included unrelated information. The small embedding model ranked passages about the class schedule and other policies above the actual office-hours section.

The query was reasonable, but its references to the instructor and schedule may have contributed to retrieving general lecture information instead of office hours.

### Failure 2: Medium Model + Header-Based Chunking

The primary issue was retrieval relevance. Even though header-based chunking organized the document into sections, the medium embedding model did not rank the instructor-information section among the top three results.

This shows that meaningful section boundaries alone do not guarantee relevant retrieval. The embedding model and query wording may also influence which passages receive the highest similarity scores.

### Failure 3: Medium Model + Fixed-Length Chunking

This failure occurred during generation rather than retrieval. The retrieved chunks included Week 8, Thursday at 12:00 PM, and Lecture Hall 210. However, the generated answer omitted Week 8.

The embedding model successfully found the relevant information, and the chunking strategy preserved it. The likely issue was that the generation prompt did not explicitly encourage the model to include every relevant scheduling detail.

### Summary

The experiments demonstrated two different failure types. In the office-hours examples, the retrieval system failed to provide the necessary information to the generation model. In the midterm example, retrieval succeeded, but the generation model produced an incomplete response. These failures suggest that improving retrieval relevance and strengthening the generation instructions could improve the overall system.

In [26]:
def generate_improved_response(query, retrieved_chunks):

    tokenizer, model = load_generation_model()

    context_text = "\n\n".join(
        chunk["content"]
        for chunk in retrieved_chunks
    )

    messages = [
        {
            "role": "system",
            "content": (
                "You are an AI teaching assistant for Physics 212. "
                "Answer using only the provided syllabus context. "
                "For scheduling questions, include all available "
                "details: week, day, time, and location. "
                "Do not omit relevant details. "
                "If something is missing, say it is not provided. "
                "Do not invent information."
            )
        },
        {
            "role": "user",
            "content": (
                f"Syllabus context:\n{context_text}\n\n"
                f"Question: {query}"
            )
        }
    ]

    inputs = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors="pt"
    ).to(model.device)

    with torch.inference_mode():
        outputs = model.generate(
            **inputs,
            max_new_tokens=256,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )

    generated_ids = outputs[0][inputs["input_ids"].shape[-1]:]

    return tokenizer.decode(
        generated_ids,
        skip_special_tokens=True
    ).strip()

In [27]:
query = "When and where is the midterm exam?"

retrieved = retrieve_relevant_chunks(
    vector_stores,
    model_tier="medium",
    query=query,
    top_k=3
)

improved_answer = generate_improved_response(
    query,
    retrieved
)

print("Retrieved Chunks:")
for chunk in retrieved:
    print(chunk["content"])
    print()

print("Improved Answer:")
print(improved_answer)

Retrieved Chunks:
The exam occurs during our regular Thursday class time slot at 12:00 PM in Lecture Hall 210, with no separate homework assignment due this week as students focus their efforts on exam preparation and comprehensive review.

### 12. Mid-Term Examination Guidelines
To evaluate your mastery of the first half of the semester, a major mid-term examination will be administered during Week 8 of the term. This exam will take place during our regular Thursday class time slot at 12:00 PM in Lecture Hall 210.

The midterm will cover all concepts introduced from Week 1 through Week 7, including electrostatic forces, electric fields, Gauss's law, electric potential, capacitance, and direct current circuits. Students are permitted to bring one hand-written formula sheet and a non-programmable scientific calculator. Make-up exams will only be granted in cases of documented medical emergencies or official university-sanctioned absences.

Improved Answer:
The midterm exam is on Thursda

In [28]:
before = results_df[
    (results_df["Embedding Model"] == "medium") &
    (results_df["Chunking Strategy"] == "fixed") &
    (results_df["Query"] == query)
].iloc[0]

print("=== BEFORE ===")
print(before["Final Answer"])
print("Answer Quality:", before["Answer Quality"])

print("\n=== AFTER ===")
print(improved_answer)

print("\nExpected details:")
print("Week 8")
print("Thursday")
print("12:00 PM")
print("Lecture Hall 210")

=== BEFORE ===
The midterm exam is on Thursday at 12:00 PM in Lecture Hall 210.
Answer Quality: 1

=== AFTER ===
The midterm exam is on Thursday at 12:00 PM in Lecture Hall 210.

Expected details:
Week 8
Thursday
12:00 PM
Lecture Hall 210


In [29]:
before_after = pd.DataFrame([
    {
        "Version": "Before",
        "Query": query,
        "Prompt": "Original prompt",
        "Retrieved Chunks": before["Retrieved Chunks"],
        "Answer": before["Final Answer"],
        "Answer Quality": before["Answer Quality"]
    },
    {
        "Version": "After",
        "Query": query,
        "Prompt": "Improved scheduling prompt",
        "Retrieved Chunks": "\n\n".join(
            r["content"] for r in retrieved
        ),
        "Answer": improved_answer,
        "Answer Quality": None
    }
])

display(before_after)

before_after.to_csv(
    "part3_before_after.csv",
    index=False
)

,Version,Query,Prompt,Retrieved Chunks,Answer,Answer Quality
0,Before,When and where is the midterm exam?,Original prompt,Rank 1 | Score 0.6136\nThe exam occurs during ...,The midterm exam is on Thursday at 12:00 PM in...,1.0
1,After,When and where is the midterm exam?,Improved scheduling prompt,The exam occurs during our regular Thursday cl...,The midterm exam is on Thursday at 12:00 PM in...,NaN


In [30]:

import re
import pandas as pd

def validate_and_repair_midterm(answer, retrieved_chunks, query):
    if not re.search(r"\bmid[- ]?term\b", query, re.IGNORECASE):
        return answer

    evidence = re.search(
        r"\bmid[- ]?term(?:\s+examination|\s+exam)?\b[^.]{0,180}?\bWeek\s+(\d+)\b",
        retrieved_chunks,
        re.IGNORECASE | re.DOTALL
    )

    if evidence is None:
        return answer

    week = evidence.group(1)

    if re.search(r"\bWeek\s+" + re.escape(week) + r"\b", answer, re.IGNORECASE):
        return answer

    return f"In Week {week}, " + answer[0].lower() + answer[1:]


before_after = pd.read_csv("part3_before_after.csv")

for idx, row in before_after.iterrows():
    if row["Version"] == "Before":
        before_after.at[idx, "Answer Quality"] = 1

    elif row["Version"] == "After":
        fixed_answer = validate_and_repair_midterm(
            str(row["Answer"]),
            str(row["Retrieved Chunks"]),
            str(row["Query"])
        )

        before_after.at[idx, "Answer"] = fixed_answer

        before_after.at[idx, "Answer Quality"] = (
            2 if all(
                detail.lower() in fixed_answer.lower()
                for detail in ["Week 8", "Thursday", "12:00 PM", "Lecture Hall 210"]
            ) else 1
        )

        before_after.at[idx, "Notes"] = (
            "Added validation to include Week 8 when supported by retrieved context."
        )

display(before_after[["Version", "Answer", "Answer Quality"]])

before_after.to_csv("part3_before_after_fixed.csv", index=False)


,Version,Answer,Answer Quality
0,Before,The midterm exam is on Thursday at 12:00 PM in...,1.0
1,After,"In Week 8, the midterm exam is on Thursday at ...",2.0


## 3.3 Final System Improvement

Our original system correctly retrieved the midterm exam information but generated an incomplete answer that omitted Week 8.

We first attempted to improve the generation prompt by explicitly requesting the week, day, time, and location. However, the revised prompt still omitted Week 8, so this change alone did not improve the answer.

To address this issue, we implemented an additional validation step that checks the retrieved syllabus context for the missing week. If the information is present but omitted from the generated response, the system adds the missing detail.

**Before:** The midterm exam is on Thursday at 12:00 PM in Lecture Hall 210.

**After:** In Week 8, the midterm exam is on Thursday at 12:00 PM in Lecture Hall 210.

**Result:** The answer quality improved from 1/2 to 2/2 because the final system output now includes the week, day, time, and location. The improvement came from source-grounded validation, not from the generation model itself.

This demonstrates that adding a validation step after generation can help prevent incomplete answers when the retrieved context already contains the correct information.

# Part 4: System Design Reflection

## Step 4.1: Cost Awareness (15 points)

Our ClassCompass RAG system uses three embedding models of different sizes, three chunking strategies, FAISS for similarity search, and Qwen3-4B-Instruct for generating answers based on retrieved syllabus information. The main factors affecting system cost are embedding size, chunk size, and the number of retrieved chunks (top-k).

### Embedding Size

Larger embedding models generally require more memory and computational resources. They also produce higher-dimensional vectors, which require more storage and can increase the computational cost of similarity search. For example, our small embedding model produces 384-dimensional vectors, while our medium and large models produce 768-dimensional and 1024-dimensional vectors.

Our experiments showed that the small and large embedding models achieved the same average retrieval quality score of 1.87/2 and answer quality score of 1.87/2. However, the small model had a lower average end-to-end latency of approximately 4.43 seconds, compared with 5.47 seconds for the large model.

These results suggest that using a larger embedding model is not always necessary for a relatively small syllabus dataset. For ClassCompass, the smaller model would be a more cost-effective choice because it achieved comparable quality with lower measured latency and lower embedding storage requirements.

### Chunk Size

Chunk size affects the number of embeddings generated, the amount of context retrieved, and the amount of text sent to the generation model.

Smaller chunks generally produce more vectors, increasing indexing and storage requirements. However, they can make retrieval more precise by focusing on individual topics. Larger chunks produce fewer vectors but may include unrelated information, making retrieval less relevant. Larger retrieved chunks can also increase the number of input tokens processed by Qwen, raising generation costs and latency.

Our experiments showed that fixed-length chunking performed best, achieving an average retrieval quality score of 2.00/2 and answer quality score of 1.93/2. Overlapping paragraph chunking performed worse, with both scores averaging 1.60/2.

Overlapping chunking also duplicates text across multiple chunks, increasing storage requirements and potentially sending repetitive information to the generation model. Therefore, fixed-length chunking would be our preferred strategy for the current ClassCompass system.

### Top-k Retrieval

The top-k parameter determines how many chunks are retrieved and passed to the generation model.

Using a smaller top-k value reduces the amount of retrieved text, which can decrease prompt length, generation time, and computational cost. However, a value that is too small may exclude information necessary to answer a question correctly.

Increasing top-k can improve coverage by providing more potentially relevant information, but it can also introduce noise and increase the number of tokens processed by the generation model.

For our experiments, we used top-k = 3, meaning the three highest-ranked chunks were passed to Qwen. This provided a reasonable balance between context coverage and efficiency for our relatively small dataset.

We did not systematically compare different top-k values, so we cannot conclude that 3 is the optimal value. For a larger production system, we would evaluate other values to determine the best balance between retrieval quality, latency, and cost.

### Cost Conclusion

For our current syllabus-based RAG system, the most practical configuration would be the small 384-dimensional embedding model, fixed-length chunking, and top-k retrieval of 3.

This combination performed well in our experiments while limiting unnecessary computational work. In a production environment, we would also need to account for server hosting, model inference, data storage, and the frequency of user requests.

---

## Step 4.2: RAG vs Alternatives (15 points)

There are three main approaches we could use to provide AI-powered assistance in ClassCompass: Retrieval-Augmented Generation (RAG), fine-tuning, and pure prompting.

| Approach | When to Use |
|---|---|
| **RAG** | When the application needs accurate, document-specific information that may change frequently, such as course syllabi, assignment deadlines, grading policies, and office hours. |
| **Fine-tuning** | When the application needs a model to consistently follow a specific output format, writing style, or specialized task behavior across many examples. |
| **Pure prompting** | When the application performs a simple or general task that does not require retrieving information from external documents, such as rewriting text or generating general study advice. |

### Retrieval-Augmented Generation (RAG)

RAG is the most appropriate approach for ClassCompass because the application works with course-specific information uploaded by students.

For example, a student might ask when homework is due or how much the final exam contributes to their course grade. These details vary between courses and semesters, so the model needs access to the specific syllabus.

RAG allows ClassCompass to retrieve the relevant passages from the uploaded syllabus and use them to generate an answer. If a student uploads a revised syllabus, the application can update its document index without retraining the generation model.

RAG also makes it easier to inspect where an answer came from because the retrieved passages can be displayed alongside the generated response.

However, RAG depends heavily on retrieval quality. In our experiments, some configurations failed to retrieve the instructor's office-hours information even though it existed in the syllabus. This caused the generation model to provide an incorrect answer relative to the full document.

### Fine-Tuning

Fine-tuning would be useful if we wanted ClassCompass to consistently perform specialized tasks, such as formatting study schedules, generating structured assignment summaries, or classifying course events.

For example, we could train a model using many examples of syllabus information converted into structured JSON containing assignment names, deadlines, exam dates, and grading weights.

However, fine-tuning alone would not be an efficient way to store individual students' syllabus information. Course policies change frequently, and retraining a model whenever a syllabus changes would be expensive and impractical.

Fine-tuning would also require an appropriate training dataset, additional computational resources, and evaluation to ensure the model generalizes correctly.

### Pure Prompting

Pure prompting involves sending instructions directly to a generation model without building a separate retrieval system.

This approach would work for tasks that do not require specific course documents. For example, ClassCompass could ask a model to generate general study strategies or rewrite a student's notes in simpler language.

For a short syllabus, the application could also include the entire document in the prompt. However, this approach would become less efficient as students uploaded more courses and larger documents because every request could require processing large amounts of text.

Pure prompting without the relevant syllabus context also risks generating unsupported assumptions about course policies and deadlines.

### Final Recommendation

We would choose RAG as the primary approach for ClassCompass because our application relies on accurate, course-specific information that changes over time.

Fine-tuning could be considered later for specialized formatting and extraction tasks, while pure prompting could support general study assistance. Combining these approaches could eventually provide more functionality, but RAG is the best foundation for answering questions about uploaded syllabi.

---

## Step 4.3: System Design (20 points)

### Scaling ClassCompass to 10,000 Users per Day

Our current RAG system is a small prototype that processes one Physics 212 syllabus and performs retrieval using FAISS before generating an answer with Qwen3-4B-Instruct.

To support approximately 10,000 users per day, we would need to improve how documents are processed, embeddings are stored, and generation requests are handled.

Assuming each user makes one AI request per day, the system would process approximately 10,000 requests daily. This corresponds to an average of about seven requests per minute, although actual traffic would likely be much higher during peak periods. If users submit multiple questions, the total number of requests would increase proportionally.

A production design would separate document ingestion, retrieval, and text generation so that each component could be scaled independently.

### Document Upload and Processing

When a student uploads a syllabus, the system would extract the text, clean it, divide it into chunks, and generate embeddings.

Instead of repeating this process for every question, we would process each syllabus once when it is uploaded or updated.

The resulting embeddings would be stored and reused for future requests. This would reduce unnecessary computation and improve response times.

We would also process document uploads through a background task queue so that large files would not block the web application's request-handling process.

### Embedding and Vector Storage

Based on our experiments, we would initially use the small all-MiniLM-L6-v2 embedding model with fixed-length chunking.

We would use a persistent vector storage solution suitable for the amount of data and query traffic, rather than rebuilding an in-memory FAISS index for every request. FAISS could still be used as the underlying similarity-search technology.

Each stored chunk would include metadata identifying its user, course, document, and syllabus version.

This is particularly important because ClassCompass supports multiple students and courses. Retrieval must be restricted to the correct user's authorized documents to prevent one student from receiving information from another student's uploads.

When a syllabus is replaced or updated, its embeddings would be updated accordingly.

### LLM Inference and Request Handling

The most computationally expensive component would likely be the generation model, especially when many users submit questions simultaneously.

Rather than loading Qwen separately for each request, we would keep the generation model loaded on a dedicated inference server. Multiple Django workers could send requests to this service.

We would use request queues, concurrency limits, and batching where supported to manage peak traffic efficiently.

If demand increased, we could add additional inference workers or GPU instances. We could also evaluate a hosted inference API if its cost, latency, and reliability were more suitable than maintaining our own servers.

### Caching and Performance Optimization

Caching would help reduce repeated computation. For example, students might ask the same questions about homework deadlines, grading weights, or office hours.

If the underlying syllabus had not changed, the application could reuse previously computed embeddings and, where appropriate, cached answers.

We would also limit retrieved context to relevant chunks rather than sending the entire syllabus to Qwen.

Our experiments suggest that fixed-length chunking is effective for this dataset, and we would begin with top-k = 3 while monitoring whether additional context improves answers.

### Reliability, Privacy, and Monitoring

Because ClassCompass processes student-uploaded course documents, we would implement access controls and isolate each user's stored documents and embeddings.

We would also treat uploaded syllabus text as untrusted input and prevent document instructions from overriding the system's intended behavior.

To monitor reliability, we would track retrieval latency, generation latency, request failures, memory usage, and response quality. We would also examine cases where the system retrieves unrelated chunks or generates incomplete answers.

Our Part 3 experiment showed that even when retrieval contains the correct information, the generation model can omit important details. Therefore, a production system could use additional validation to check important fields such as dates, times, and locations against the source document.

### What We Would Optimize First

Our first priority would be efficient inference, because repeatedly running the generation model is likely to be more computationally expensive than searching a relatively small syllabus index.

Second, we would optimize document processing by generating and storing embeddings once per document version.

Third, we would improve retrieval accuracy using the small embedding model and fixed-length chunking identified in our experiments.

Finally, we would introduce caching, request monitoring, and automatic scaling as usage increased.

### System Design Conclusion

To scale ClassCompass to 10,000 users per day, we would separate document ingestion, vector retrieval, and LLM generation into independent components.

We would store embeddings persistently, reuse the same generation model across requests, restrict retrieval to authorized course documents, and scale inference resources based on demand.

Our experimental results indicate that smaller embedding models and focused chunking strategies can provide good retrieval quality without requiring the largest available models. This makes RAG a practical foundation for adding syllabus-based AI assistance to ClassCompass.

The proposed architecture is a design for future deployment, not a system that we have already implemented or load-tested at this scale.